In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Sentiment Analysis of Foreign Affairs Texts

## Overview

This notebook applies the sentiment analysis pipeline to the manifesto sentences previously identified as belonging to the **Foreign Affairs** policy dimension.

The topic analysis notebook produces an intermediate corpus containing manifesto text classified as relevant to Foreign Affairs. This notebook takes that corpus as input and estimates the sentiment expressed towards countries mentioned in the corresponding text.

The analysis is performed separately for the **United States** and **Australia**.

### Analytical pipeline

The complete workflow is:

```text
Foreign Affairs manifesto sentences
              │
              ▼
      Sentiment analysis
              │
              ▼
Sentiment score for each text segment
              │
              ▼
       Country detection
              │
              ▼
Sentiment associated with each country
              │
              ▼
Aggregation by party / election / country
              │
              ▼
       Visualisation
```

The resulting country-level scores can then be used to study how political parties express positive or negative sentiment towards specific countries in their Foreign Affairs discourse.

In [ ]:
import pandas as pd
import spacy
from sentiment_analysis import sentiment_analysis as sa
from sentiment_analysis.country_aliases_generation import generate_wikidata_country_dictionary

# Global sentiment analysis

## 3. Global configuration

The sentiment pipeline relies on the English spaCy model.

A dictionary of country aliases is also generated. This dictionary is used to recognise references to countries even when they do not appear under a single canonical name.

For example, a country can potentially be mentioned using different forms, abbreviations or alternative names. Mapping these aliases to a common country identifier makes it possible to aggregate sentiment consistently.

The sentiment analyser is then initialised using both the spaCy model and the country dictionary.

At this stage, the NLP components required by the rest of the pipeline are ready.

In [ ]:
NLP = spacy.load("en_core_web_sm")
COUNTRY_SYNONYMS_DICT = generate_wikidata_country_dictionary()

sentiment_analyzer = sa.setup_sentiment_analyzer(
    nlp=NLP, 
    country_synonyms_dict=COUNTRY_SYNONYMS_DICT
)

# 2. USA manifesto analysis

The first step loads the `Foreign_Affairs` corpus extracted during the topic-analysis stage.

The sentiment score is then computed for the texts in the corpus. This produces the sentiment information required for the subsequent country-level analysis.

The overall sentiment distribution can be visualized to inspect the results before focusing on individual countries.

In [ ]:
usa_foreignAffairs = pd.read_csv("../data/intermediate/foreign_affairs/usa_foreignAffairs_text.csv", index_col=0)
usa_foreignAffairs = sa.compute_sentiment(usa_foreignAffairs)

In [ ]:
sa.plot_sentiment(usa_foreignAffairs)

In [ ]:
foreignAffairs_usa = sa.compute_country_scores(
    section_dataframe=usa_foreignAffairs, 
    nlp=NLP, 
    analyzer=sentiment_analyzer, 
    country_synonyms_dict=COUNTRY_SYNONYMS_DICT
)

## 3. Country-level sentiment

The next step identifies countries mentioned in the manifesto sentences and associates the corresponding sentiment scores with these countries.

This allows the analysis to move from general sentiment towards a more specific question:

> What sentiment is expressed towards a given country when it is discussed in the Foreign Affairs sections of political manifestos?

The country-level scores retain the political party and election information, making it possible to compare how different parties discuss the same country over time.

For example, the analysis below extracts the sentiment associated with **Australia** in US manifestos.


In [ ]:
sa.plot_country_sentiment(foreignAffairs_usa, "Democratic Party", 2008, figsize=(10, 12))

In [ ]:
target_country = "AUSTRALIA"
australia_scores = sa.get_country_sentiment(foreignAffairs_usa, country=target_country)

In [ ]:
australia_scores

## 4. Australia manifesto analysis

The same procedure is applied to Australian manifestos.

The `Foreign_Affairs` corpus generated during the topic-analysis stage is loaded and passed through the sentiment-analysis pipeline.

Country mentions are then identified and aggregated to obtain country-specific sentiment scores.

This makes it possible, for example, to study how the **United States** is discussed by Australian political parties.

In [ ]:

aus_foreignAffairs = pd.read_csv("../data/intermediate/foreign_affairs/australia_foreignAffairs_text.csv", index_col=0)
aus_foreignAffairs = sa.compute_sentiment(aus_foreignAffairs)

In [ ]:
sa.plot_sentiment(aus_foreignAffairs)

In [ ]:
foreignAffairs_aus = sa.compute_country_scores(
    section_dataframe=aus_foreignAffairs, 
    nlp=NLP, 
    analyzer=sentiment_analyzer, 
    country_synonyms_dict=COUNTRY_SYNONYMS_DICT
)

In [ ]:
sa.plot_country_sentiment(foreignAffairs_aus, "Australian Labor Party", 1972, figsize=(10, 12))

In [ ]:
target_country = "UNITED_STATES"
usa_scores = sa.get_country_sentiment(foreignAffairs_aus, country=target_country)

In [ ]:
usa_scores

## 5. Interpretation

The final outputs of this notebook are country-level sentiment scores associated with a political party and an election year.

These scores constitute the final analytical data used to compare the treatment of foreign countries across parties, elections and countries.

The interpretation should therefore consider both:

* **topic selection**, which determines which sentences enter the analysis;
* **sentiment scoring**, which determines the tone associated with the countries mentioned in those sentences.

The sentiment analysis should consequently be understood as the second stage of the broader pipeline rather than as an independent analysis.